In [1]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 

ROOT_DIR = '../..'
sys.path.append(f'{ROOT_DIR}/src')


import openai
import os

# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
import json
with open(f"{ROOT_DIR}/API_KEYS2.json", "r") as file:
    api_keys = json.load(file)

os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
os.environ['CACHE_DIR'] = os.path.join(ROOT_DIR, 'cache_dir3')

# MassMaps

In [2]:
import torch
from datasets import load_dataset

test_dataset = load_dataset("BrachioLab/massmaps-cosmogrid-100k", split='test')
test_dataset.set_format('torch', columns=['input', 'label'])

In [3]:
# import importlib
import sys; sys.path.append("../src")
# import massmaps
# importlib.reload(massmaps)
from massmaps import MassMapsExample
from massmaps import massmap_to_pil_norm, get_llm_generated_answer, get_llm_output
from massmaps import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores
from llms import load_model

In [4]:
from tqdm.auto import tqdm
import json

In [5]:
# model = 'gpt-4o'
models = [
    'gpt-4o',
    # 'claude-3-5-sonnet-latest',
    # 'gemini-2.0-flash',
    # 'o1'
]

eval_model_name = 'gpt-4o'
eval_model = load_model(eval_model_name)



In [6]:
methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

In [7]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
from massmaps import isolate_individual_features_expert

In [9]:
import json
import copy
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        load_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}_{eval_model_name}.json')

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = 3 #len(results)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            example_dict = result
            
            example = MassMapsExample(
                input = torch.tensor(example_dict['input']).to(device),
                answer = example_dict['answer'],
                llm_answer = example_dict['llm_answer'],
                llm_explanation = example_dict['llm_explanation'],
            )
            
            # isolate individual features
            expert_claims = isolate_individual_features_expert(
                example.llm_explanation, model=eval_model
            )
            
            if expert_claims is None:
                continue
            
            example.expert_claims = [claim.strip() for claim in expert_claims]
            
            claims = isolate_individual_features(
                example.llm_explanation, model=eval_model
            )
            
            if claims is None:
                continue
            
            example.claims = [claim.strip() for claim in claims]
            
            new_results.append(example)
            
#             # distill relevant features
#             relevant_claims = distill_relevant_features(
#                 example.input, 
#                 example.llm_answer,
#                 example.claims,
#                 model=eval_model
#             )
#             example.relevant_claims = relevant_claims

#             # calculate expert alignment scores
#             align_infos = calculate_expert_alignment_scores(
#                 example.relevant_claims, 
#                 eval_model,
#             )

#             alignable_claims = [info["Claim"] for info in align_infos]
#             alignment_categories = [info["Category"] for info in align_infos]
#             aligned_category_ids = [info["Category ID"] for info in align_infos]
#             alignment_scores = [info["Alignment"] for info in align_infos]
#             alignment_raws = [info["Alignment Raw"] for info in align_infos]
#             alignment_reasonings = [info["Reasoning"] for info in align_infos]
            
#             example.alignable_claims = alignable_claims
#             example.alignment_categories = alignment_categories
#             example.aligned_category_ids = aligned_category_ids
#             example.alignment_scores = alignment_scores
#             example.alignment_raws = alignment_raws
#             example.alignment_reasonings = alignment_reasonings
            
#             # save
#             save_dict = {}
#             for k, v in example.__dict__.items():
#                 save_dict[k] = v if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
#             # with open(save_path, 'wt') as output_file:
#             #     json.dump(save_dict, output_file)

#             new_results.append(save_dict)


#         with open(save_path, 'wt') as output_file:
#             json.dump(new_results, output_file, indent=4)

=== Using model gpt-4o ===
=== Using method vanilla ===


  0%|          | 0/3 [00:00<?, ?it/s]

In [20]:
# === Full fuzzy highlighting (softer matching, sentence-scoped, tqdm, multi-overlap) ===
import re
from difflib import SequenceMatcher
from dataclasses import dataclass
from typing import List, Tuple, Optional
import pandas as pd
from tqdm.notebook import tqdm

try:
    from IPython.display import display, HTML
    _HAS_IPY = True
except Exception:
    _HAS_IPY = False

# ---------- Config ----------
DEFAULT_MIN_RATIO = 0.50        # start here
BACKOFF_MIN_RATIO = 0.40        # try this if no match found
WINDOW_SCALE = (0.50, 1.80)     # min/max window len as fraction of pattern len (after normalization)

# ---------- Data ----------
@dataclass
class SpanMatch:
    label: str
    text: str
    start: int
    end: int
    score: float
    kind: str

# ---------- Utils ----------
def _normalize(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^\w\s\.\-\+_']", " ", s)  # keep word-ish chars, dots, +, - (helps Omega_m, sigma_8)
    s = re.sub(r"\s+", " ", s)
    return s

_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def split_sentences_with_offsets(text: str) -> List[Tuple[int, int, str]]:
    spans = []
    start = 0
    for m in _SENT_SPLIT.finditer(text):
        end = m.start()
        if end > start:
            spans.append((start, end, text[start:end]))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text), text[start:]))
    return spans

def _strip_category_prefix(s: str) -> str:
    # drop "1. Title: " style prefixes if present
    return re.sub(r"^\s*\d+\.\s*[^:]{0,80}:\s*", "", s).strip()

# ---------- Matching ----------
def _best_in_bounds(s: str, pattern: str, min_ratio: float, bounds: Tuple[int,int]) -> Optional[Tuple[int,int,float]]:
    lo, hi = bounds
    if lo >= hi or not pattern.strip():
        return None

    # Build normalized slice + index map to original
    norm_chars, map_norm_to_orig, in_space = [], [], False
    for idx, ch in enumerate(s[lo:hi], start=lo):
        if ch.isspace():
            if not in_space:
                norm_chars.append(' ')
                map_norm_to_orig.append(idx)
                in_space = True
        else:
            norm_chars.append(ch.lower())
            map_norm_to_orig.append(idx)
            in_space = False

    if not norm_chars:
        return None

    # trim leading/trailing spaces
    L = 0
    while L < len(norm_chars) and norm_chars[L] == ' ': L += 1
    R = len(norm_chars) - 1
    while R >= 0 and norm_chars[R] == ' ': R -= 1
    if R < L: return None

    norm_to_orig = map_norm_to_orig[L:R+1]
    s_norm2 = "".join(norm_chars[L:R+1])
    p_norm  = _normalize(pattern)
    if not s_norm2 or not p_norm:
        return None

    tlen = max(3, len(p_norm))
    wmin = max(3, int(round(tlen * WINDOW_SCALE[0])))
    wmax = min(len(s_norm2), int(round(tlen * WINDOW_SCALE[1])))

    # Prefer higher score, then shorter window, then later start
    best_tuple = (0.0, float("-inf"), -1)
    best_range = (None, None)

    for wlen in range(wmin, wmax + 1):
        for start in range(0, len(s_norm2) - wlen + 1):
            ratio = SequenceMatcher(None, s_norm2[start:start + wlen], p_norm).ratio()
            key = (ratio, -wlen, start)
            if key > best_tuple:
                best_tuple, best_range = key, (start, start + wlen)

    score = best_tuple[0]
    if score < min_ratio or best_range[0] is None:
        return None

    s0_norm, s1_norm = best_range
    start_orig = norm_to_orig[s0_norm]
    end_orig   = norm_to_orig[s1_norm - 1] + 1

    # Expand to token boundaries but don't cross punctuation
    is_left  = lambda i: i == 0 or re.match(r"[\s,;:()\-–—]", s[i-1] if i > 0 else "")
    is_right = lambda i: i >= len(s) or re.match(r"[\s,;:()\-–—.?!]", s[i:i+1])
    while start_orig > lo and not is_left(start_orig):  start_orig -= 1
    while end_orig < hi and not is_right(end_orig):     end_orig   += 1

    return start_orig, end_orig, score

def best_fuzzy_substring(s: str, pattern: str, min_ratio: float, bounds: Tuple[int,int]) -> Optional[Tuple[int,int,float]]:
    # try with min_ratio; if no hit, back off slightly
    res = _best_in_bounds(s, pattern, min_ratio, bounds)
    if res is None and min_ratio > BACKOFF_MIN_RATIO:
        res = _best_in_bounds(s, pattern, BACKOFF_MIN_RATIO, bounds)
    return res

def match_spans(original_text: str, phrases: List[str], kind: str,
                min_ratio: float = DEFAULT_MIN_RATIO, prefix: str = "C",
                scope: str = "per_sentence") -> List[SpanMatch]:
    matches = []
    sent_spans = split_sentences_with_offsets(original_text) if scope == "per_sentence" else [(0, len(original_text), original_text)]

    for i, phrase in enumerate(tqdm(phrases, desc=f"Matching {kind}s", leave=False), start=1):
        candidate = phrase.strip()
        if kind == "expert":
            if re.search(r"\bN/?A\b", candidate, flags=re.I):  # skip N/A rows
                continue
            candidate = _strip_category_prefix(candidate)
        if not candidate:
            continue

        best, best_score = None, -1.0
        for s_lo, s_hi, _ in sent_spans:
            hit = best_fuzzy_substring(original_text, candidate, min_ratio, (s_lo, s_hi))
            if hit:
                a, b, sc = hit
                # prefer higher score; if tie, shorter span
                if sc > best_score or (abs(sc - best_score) < 1e-9 and (b - a) < (best[1] - best[0])):
                    best, best_score = (a, b), sc
        if best:
            s0, s1 = best
            matches.append(SpanMatch(f"{prefix}{i}", phrase, s0, s1, best_score, kind))
    return matches

# ---------- Rendering (multi-claim overlaps with stripes) ----------
def _striped_background(colors: List[str]) -> str:
    if not colors: return ""
    if len(colors) == 1: return f"background:{colors[0]}33;"
    n = len(colors)
    stops = []
    for i, c in enumerate(colors):
        a = int(i * 100 / n); b = int((i + 1) * 100 / n)
        stops.append(f"{c}33 {a}% {b}%")
    return f"background-image: repeating-linear-gradient(135deg, {', '.join(stops)});"

def render_highlights(original_text: str, matches: List[SpanMatch], palette: List[str], title: str) -> str:
    N = len(original_text)
    if not matches:
        return f"<div><h3>{title}</h3><p>{original_text}</p><p><em>No matches.</em></p></div>"

    owners = [list() for _ in range(N)]
    for idx, m in enumerate(matches):
        s0, s1 = max(0, m.start), min(N, m.end)
        for j in range(s0, s1):
            owners[j].append(idx)

    html = [f"<div style='font-family:system-ui;line-height:1.5;max-width:1000px'>",
            f"<h3>{title}</h3>",
            "<div style='padding:10px;border:1px solid #ddd;border-radius:8px'>"]

    i = 0
    while i < N:
        if original_text[i] == "\n":
            html.append("<br/>"); i += 1; continue
        key = tuple(sorted(owners[i]))
        j = i + 1
        while j < N and original_text[j] != "\n" and tuple(sorted(owners[j])) == key:
            j += 1
        seg = original_text[i:j]
        if not key:
            html.append(HTML.escape(seg) if hasattr(HTML, 'escape') else seg.replace("<","&lt;").replace(">","&gt;"))
        else:
            colors = [palette[k % len(palette)] for k in key]
            labels = [matches[k].label for k in key]
            tooltip = ", ".join(labels)
            bg_css = _striped_background(colors)
            html.append(
                f"<span title='{tooltip}' style='border-bottom:2px solid {colors[0]};"
                f"border-radius:4px;padding:0 2px;{bg_css}'>{seg.replace('<','&lt;').replace('>','&gt;')}</span>"
            )
        i = j

    html.append("</div><div style='margin-top:8px;font-size:0.95em'><strong>Legend</strong><ul>")
    for k, m in enumerate(matches):
        color = palette[k % len(palette)]
        html.append(
            f"<li><span style='display:inline-block;width:10px;height:10px;background:{color};margin-right:6px'></span>"
            f"<strong>{m.label}</strong> ({m.kind}, score={m.score:.2f}): {m.text}</li>"
        )
    html.append("</ul></div></div>")
    return "".join(html)

def matches_to_df(matches: List[SpanMatch], original_text: str) -> pd.DataFrame:
    return pd.DataFrame([{
        "label": m.label, "kind": m.kind, "score": round(m.score, 3),
        "start": m.start, "end": m.end,
        "matched_text": original_text[m.start:m.end],
        "claim_text": m.text
    } for m in matches]).sort_values(["kind", "label"])


In [21]:
di = 0
print('===== LLM EXPLANATION =====')
print(new_results[di].llm_explanation)

print('===== CLAIMS =====')
for claim in new_results[di].claims:
    print(claim)

print('===== EXPERT CLAIMS =====')
for claim in new_results[di].expert_claims:
    print(claim)

# === Example usage (unchanged) ===
di = 0
original = new_results[di].llm_explanation
claims = list(new_results[di].claims)
expert_claims = [str(x) for x in new_results[di].expert_claims]

# Tune min_ratio to control match strictness
claim_matches  = match_spans(original, claims, kind="claim",  min_ratio=0.6, prefix="C")
expert_matches = match_spans(original, expert_claims, kind="expert", min_ratio=0.6, prefix="E")

palette_claims  = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#edc948", "#b07aa1", "#ff9da7", "#9c755f", "#bab0ab"]
palette_experts = ["#3b82f6", "#22c55e", "#ef4444", "#a855f7", "#06b6d4", "#eab308", "#f97316", "#84cc16", "#f43f5e", "#14b8a6"]

html_claims  = render_highlights(original, claim_matches,  palette_claims,  title="Original sentence with CLAIM spans")
html_experts = render_highlights(original, expert_matches, palette_experts, title="Original sentence with EXPERT CLAIM spans")

if _HAS_IPY:
    display(HTML(html_claims))
    display(HTML(html_experts))
    display(matches_to_df(claim_matches + expert_matches, original))
else:
    print(html_claims)
    print("\n" + "="*80 + "\n")
    print(html_experts)
    print("\nMatch table (CSV):\n")
    print(matches_to_df(claim_matches + expert_matches, original).to_csv(index=False))


===== LLM EXPLANATION =====
The weak lensing map shows a mixture of colors with a dominant presence of gray and red, indicating regions with mass density fluctuations around and above zero. Some yellow regions suggest areas with higher peaks of density exceeding 2.9 standard deviations. The presence of these structures indicates a relatively rich and varied mass distribution, likely pointing to a moderate value of both Omega_m and sigma_8.
===== CLAIMS =====
The weak lensing map shows a mixture of colors with a dominant presence of gray and red.
Gray regions indicate mass density fluctuations around zero.
Red regions indicate mass density fluctuations above zero.
Some yellow regions suggest areas with higher peaks of density exceeding 2.9 standard deviations.
The presence of gray, red, and some yellow structures indicates a relatively rich and varied mass distribution.
A relatively rich and varied mass distribution likely points to a moderate value of both Omega_m and sigma_8.
===== EX

Matching claims:   0%|          | 0/6 [00:00<?, ?it/s]

Matching experts:   0%|          | 0/7 [00:00<?, ?it/s]

,label,kind,score,start,end,matched_text,claim_text
0,C1,claim,0.994,0,87,The weak lensing map shows a mixture of colors...,The weak lensing map shows a mixture of colors...
1,C2,claim,0.852,89,149,indicating regions with mass density fluctuati...,Gray regions indicate mass density fluctuation...
2,C3,claim,0.817,89,161,indicating regions with mass density fluctuati...,Red regions indicate mass density fluctuations...
3,C4,claim,1.000,162,259,Some yellow regions suggest areas with higher ...,Some yellow regions suggest areas with higher ...
4,C5,claim,0.879,260,349,The presence of these structures indicates a r...,"The presence of gray, red, and some yellow str..."
5,C6,claim,0.977,303,415,a relatively rich and varied mass distribution...,A relatively rich and varied mass distribution...
6,E1,expert,0.770,162,259,Some yellow regions suggest areas with higher ...,1. Lensing Peak (Cluster) Abundance: Some yell...
7,E4,expert,0.427,260,415,The presence of these structures indicates a r...,4. Fine-Scale Clumpiness: The dominant presenc...
8,E6,expert,0.735,27,161,a mixture of colors with a dominant presence o...,6. Density Contrast Extremes: The mixture of c...


In [22]:
di = 1
print('===== LLM EXPLANATION =====')
print(new_results[di].llm_explanation)

print('===== CLAIMS =====')
for claim in new_results[di].claims:
    print(claim)

print('===== EXPERT CLAIMS =====')
for claim in new_results[di].expert_claims:
    print(claim)

# === Example usage (unchanged) ===
original = new_results[di].llm_explanation
claims = list(new_results[di].claims)
expert_claims = [str(x) for x in new_results[di].expert_claims]

# Tune min_ratio to control match strictness
claim_matches  = match_spans(original, claims, kind="claim",  min_ratio=0.6, prefix="C")
expert_matches = match_spans(original, expert_claims, kind="expert", min_ratio=0.6, prefix="E")

palette_claims  = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#edc948", "#b07aa1", "#ff9da7", "#9c755f", "#bab0ab"]
palette_experts = ["#3b82f6", "#22c55e", "#ef4444", "#a855f7", "#06b6d4", "#eab308", "#f97316", "#84cc16", "#f43f5e", "#14b8a6"]

html_claims  = render_highlights(original, claim_matches,  palette_claims,  title="Original sentence with CLAIM spans")
html_experts = render_highlights(original, expert_matches, palette_experts, title="Original sentence with EXPERT CLAIM spans")

if _HAS_IPY:
    display(HTML(html_claims))
    display(HTML(html_experts))
    display(matches_to_df(claim_matches + expert_matches, original))
else:
    print(html_claims)
    print("\n" + "="*80 + "\n")
    print(html_experts)
    print("\nMatch table (CSV):\n")
    print(matches_to_df(claim_matches + expert_matches, original).to_csv(index=False))


===== LLM EXPLANATION =====
The weak lensing map shows a mix of underdense regions (blue) and overdense regions (red and yellow), indicating matter inhomogeneities. The presence of significant red areas suggests moderate fluctuations. The yellow spots indicate highly overdense regions, suggesting higher fluctuation amplitude. The distribution and intensity of these colors imply moderate matter density and fluctuations.
===== CLAIMS =====
The weak lensing map shows a mix of underdense regions and overdense regions, indicated by blue and red and yellow colors, respectively.
The colors in the map indicate matter inhomogeneities.
The presence of significant red areas suggests moderate fluctuations.
The yellow spots in the map indicate highly overdense regions.
The yellow spots suggest a higher fluctuation amplitude.
The distribution and intensity of underdense and overdense regions imply moderate matter density and fluctuations.
===== EXPERT CLAIMS =====
1. Lensing Peak (Cluster) Abundance

Matching claims:   0%|          | 0/6 [00:00<?, ?it/s]

Matching experts:   0%|          | 0/7 [00:00<?, ?it/s]

,label,kind,score,start,end,matched_text,claim_text
0,C1,claim,0.785,0,112,The weak lensing map shows a mix of underdense...,The weak lensing map shows a mix of underdense...
1,C2,claim,0.729,93,136,"yellow), indicating matter inhomogeneities.",The colors in the map indicate matter inhomoge...
2,C3,claim,1.000,137,206,The presence of significant red areas suggests...,The presence of significant red areas suggests...
3,C4,claim,0.893,207,257,The yellow spots indicate highly overdense reg...,The yellow spots in the map indicate highly ov...
4,C5,claim,0.804,240,299,"overdense regions, suggesting higher fluctuati...",The yellow spots suggest a higher fluctuation ...
5,C6,claim,0.856,300,394,The distribution and intensity of these colors...,The distribution and intensity of underdense a...
6,E1,expert,0.798,207,299,The yellow spots indicate highly overdense reg...,1. Lensing Peak (Cluster) Abundance: The yello...
7,E2,expert,0.618,137,206,The presence of significant red areas suggests...,2. Void Size and Frequency: The presence of bl...
8,E4,expert,0.673,137,206,The presence of significant red areas suggests...,4. Fine-Scale Clumpiness: The presence of sign...
9,E5,expert,0.966,9,136,lensing map shows a mix of underdense regions ...,5. Connectivity of the Cosmic Web: The map sho...


In [23]:
di = 2
print('===== LLM EXPLANATION =====')
print(new_results[di].llm_explanation)

print('===== CLAIMS =====')
for claim in new_results[di].claims:
    print(claim)

print('===== EXPERT CLAIMS =====')
for claim in new_results[di].expert_claims:
    print(claim)

# === Example usage (unchanged) ===
original = new_results[di].llm_explanation
claims = list(new_results[di].claims)
expert_claims = [str(x) for x in new_results[di].expert_claims]

# Tune min_ratio to control match strictness
claim_matches  = match_spans(original, claims, kind="claim",  min_ratio=0.6, prefix="C")
expert_matches = match_spans(original, expert_claims, kind="expert", min_ratio=0.6, prefix="E")

palette_claims  = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#edc948", "#b07aa1", "#ff9da7", "#9c755f", "#bab0ab"]
palette_experts = ["#3b82f6", "#22c55e", "#ef4444", "#a855f7", "#06b6d4", "#eab308", "#f97316", "#84cc16", "#f43f5e", "#14b8a6"]

html_claims  = render_highlights(original, claim_matches,  palette_claims,  title="Original sentence with CLAIM spans")
html_experts = render_highlights(original, expert_matches, palette_experts, title="Original sentence with EXPERT CLAIM spans")

if _HAS_IPY:
    display(HTML(html_claims))
    display(HTML(html_experts))
    display(matches_to_df(claim_matches + expert_matches, original))
else:
    print(html_claims)
    print("\n" + "="*80 + "\n")
    print(html_experts)
    print("\nMatch table (CSV):\n")
    print(matches_to_df(claim_matches + expert_matches, original).to_csv(index=False))


===== LLM EXPLANATION =====
The map displays a mix of colors, with gray indicating average density, red suggesting higher matter densities, and blue indicating lower densities. The presence of several yellow spots, which are above 2.9 standard deviations, suggests significant fluctuations. The mix of colors and notable structures like filament-like patterns imply a moderately heterogeneous matter distribution.
===== CLAIMS =====
The map displays a mix of colors.
Gray indicates average density in the map.
Red suggests higher matter densities in the map.
Blue indicates lower densities in the map.
The presence of several yellow spots suggests significant fluctuations.
The yellow spots are above 2.9 standard deviations.
The mix of colors and notable structures like filament-like patterns imply a moderately heterogeneous matter distribution.
===== EXPERT CLAIMS =====
1. Lensing Peak (Cluster) Abundance: The presence of several yellow spots, which are above 2.9 standard deviations, suggests 

Matching claims:   0%|          | 0/7 [00:00<?, ?it/s]

Matching experts:   0%|          | 0/7 [00:00<?, ?it/s]

,label,kind,score,start,end,matched_text,claim_text
0,C1,claim,0.985,0,32,The map displays a mix of colors,The map displays a mix of colors.
1,C2,claim,0.785,39,86,"gray indicating average density, red suggesting",Gray indicates average density in the map.
2,C3,claim,0.831,72,115,"red suggesting higher matter densities, and",Red suggests higher matter densities in the map.
3,C4,claim,0.784,116,148,blue indicating lower densities.,Blue indicates lower densities in the map.
4,C5,claim,0.772,149,262,"The presence of several yellow spots, which ar...",The presence of several yellow spots suggests ...
5,C6,claim,0.895,165,226,"several yellow spots, which are above 2.9 stan...",The yellow spots are above 2.9 standard deviat...
6,C7,claim,1.000,263,385,The mix of colors and notable structures like ...,The mix of colors and notable structures like ...
7,E1,expert,0.851,149,262,"The presence of several yellow spots, which ar...",1. Lensing Peak (Cluster) Abundance: The prese...
8,E2,expert,0.528,34,110,"with gray indicating average density, red sugg...",2. Void Size and Frequency: The blue indicatin...
9,E3,expert,0.919,274,385,colors and notable structures like filament-li...,3. Filament Thickness and Sharpness: The menti...
